Error analysis from combined_results.json

In [ ]:
import csv
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

UNKNOWN = "UNKNOWN"
COMBINED_PATH = "../outputs/notebook/experiments/t1_proto_s32_k3_km_on5k_tu085_tn040_omc25/eval/eval/combined_results.json"
OUT_DIR = "../outputs/notebook/experiments/t1_proto_s32_k3_km_on5k_tu085_tn040_omc25/eval/eval"
TOPK_CLASSES = 25
TOPN = 25

In [ ]:
combined_path = Path(COMBINED_PATH).expanduser().resolve()
out_dir = Path(OUT_DIR).expanduser().resolve()
out_dir.mkdir(parents=True, exist_ok=True)

data = json.loads(combined_path.read_text())
preds = data["predictions"]
print(f"{len(preds)} predictions → {out_dir}")

In [ ]:
def label_of(v):
    return str(v if v is not None else "").strip() or "UNLABELED"

def true_pred(p):
    return label_of(p.get("true_bucket", p.get("true_label"))), label_of(p.get("pred_label"))

def top_names(counter, topk=TOPK_CLASSES):
    names = [c for c, _ in counter.most_common(max(2, int(topk)))]
    if UNKNOWN in counter and UNKNOWN not in names:
        names.append(UNKNOWN)
    return sorted(set(names), key=lambda x: (x != UNKNOWN, x))

def save_show(fig, name):
    fig.tight_layout()
    path = out_dir / Path(name).with_suffix(".pdf")
    fig.savefig(path, format="pdf")
    plt.show()

def save_json(name, obj):
    (out_dir / name).write_text(json.dumps(obj, indent=2))

In [ ]:
# Confusion matrix (row-normalized)
names = top_names(Counter(true_pred(p)[0] for p in preds))
idx = {n: i for i, n in enumerate(names)}
mat = np.zeros((len(names), len(names)), dtype=np.int64)
for p in preds:
    t, r = true_pred(p)
    if t in idx and r in idx:
        mat[idx[t], idx[r]] += 1

mat_norm = mat / np.maximum(mat.sum(axis=1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(mat_norm, cmap="Blues", vmin=0, vmax=1)
ax.set(title="Confusion Matrix (row-normalized)", xlabel="Predicted", ylabel="True bucket")
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=90, fontsize=8)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
save_show(fig, "confusion_matrix_row_normalized.pdf")

with (out_dir / "confusion_matrix_counts.csv").open("w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["true\\pred"] + names)
    for i, n in enumerate(names):
        w.writerow([n] + mat[i].tolist())

In [ ]:
# Per-class precision / recall
tp, pred_count, true_count = Counter(), Counter(), Counter()
for p in preds:
    t, r = true_pred(p)
    true_count[t] += 1; pred_count[r] += 1
    if t == r:
        tp[t] += 1

classes = top_names(true_count)
prec = [tp[c] / max(1, pred_count[c]) for c in classes]
rec = [tp[c] / max(1, true_count[c]) for c in classes]

x, w = np.arange(len(classes)), 0.4
fig, ax = plt.subplots(figsize=(max(10, len(classes) * 0.5), 6))
ax.bar(x - w/2, prec, width=w, label="Precision")
ax.bar(x + w/2, rec, width=w, label="Recall")
ax.set(ylim=(0, 1), ylabel="Score", title="Per-Class Precision/Recall")
ax.set_xticks(x); ax.set_xticklabels(classes, rotation=75, ha="right", fontsize=8)
ax.legend(); save_show(fig, "per_class_precision_recall.pdf")

with (out_dir / "per_class_precision_recall.csv").open("w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["class", "tp", "pred_count", "true_count", "precision", "recall"])
    for i, c in enumerate(classes):
        w.writerow([c, tp[c], pred_count[c], true_count[c], prec[i], rec[i]])

In [ ]:
# Top FP classes + confused pairs
topn = max(5, int(TOPN))
fp, confused = Counter(), Counter()
for p in preds:
    t, r = true_pred(p)
    if t == UNKNOWN and r != UNKNOWN:
        fp[r] += 1
    elif t != UNKNOWN and r != UNKNOWN and t != r:
        confused[(t, r)] += 1

top_fp, top_pairs = fp.most_common(topn), confused.most_common(topn)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
if top_fp:
    lbl, val = zip(*top_fp)
    axes[0].barh(list(lbl)[::-1], list(val)[::-1])
axes[0].set(title="Top FP (unknown GT → known pred)", xlabel="count")
if top_pairs:
    axes[1].barh([f"{a} -> {b}" for (a, b), _ in top_pairs][::-1], [c for _, c in top_pairs][::-1])
axes[1].set(title="Most confused pairs", xlabel="count")
save_show(fig, "top_fp_and_confused_pairs.pdf")
save_json("top_false_positive_classes.json", top_fp)
save_json("top_confused_class_pairs.json", [{"true": a, "pred": b, "count": c} for (a, b), c in top_pairs])